In [0]:
#test second commit

#Duplicate rows after join

In [0]:
data1=[101,101,102]
data2=[101,101,103]
df1 = spark.createDataFrame(data1,['id'])
df2 = spark.createDataFrame(data2, ['id'])
# df1.join(df2, 'id').show()
# Deduplicate
df1.dropDuplicates().join(df2.dropDuplicates(), 'id').show()

![image_1776186804257.png](./image_1776186804257.png "image_1776186804257.png")

#2. Deduplication in join using latest record 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
win=Window.partitionBy('id').orderBy(col('trx_date').desc())
data1=[{'id':1,'trx_date':'2026-04-10 10:00:00','status':'active' },
{'id':1,'trx_date':'2026-04-11 09:00:00','status':'inactive'}, 
{'id':2,'trx_date':'2026-04-10 08:00:00','status':'active'}]
data2=[{'id':1},{'id':2}]
df1 = spark.createDataFrame(data1)
df2 = spark.createDataFrame(data2)
df1_ddu=df1.withColumn('rn',row_number().over(win)).filter('rn=1').drop('rn')
df1_ddu.display()
# df1.join(df2, 'id').show()



#3. Null values in join key 

In [0]:
df1=spark.createDataFrame([1,2,None],['id'])
df2=spark.createDataFrame([1,None,3],['id'])
# Here Null value does not amtch to null
df1.join(df2,'id','left').display()

# Here it will do 
from pyspark.sql.functions import col
df1.join(df2, df1.id.eqNullSafe(df2.id), 'left').display()


#4. Left join with nulls on right side 

In [0]:
data1=[ 
{'cust_id':1,'country_code':'US'}, 
{'cust_id':2,'country_code':'IN'}, 
{'cust_id':3,'country_code':None}]

data2=[{'country_code':'US','country_name':'UnitedState'}, 
{'country_code':'IN','country_name':'India'}]
df1=spark.createDataFrame(data1)
df2=spark.createDataFrame(data2)
df1.join(df2, df1.country_code==(df2.country_code), 'left').withColumn("country_name", coalesce("country_name", lit("Unknown"))) .display()



In [0]:
from pyspark.sql.functions import coalesce, lit 

help(coalesce())

#5. No data in target table 

In [0]:
df = spark.createDataFrame([
    ("A", "2026-04-10"),
    ("B", "2026-04-12"),
    ("C", "2026-04-15")
], ["id", "updated_at"])
target_df = spark.createDataFrame([], df.schema)


In [0]:
from pyspark.sql.functions import max

last_processed = target_df.agg(max("updated_at")).collect()[0][0]
print(last_processed)

In [0]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("updated_at") > last_processed)
filtered_df.show()

In [0]:
if last_processed is None:
    filtered_df = df
else:
    filtered_df = df.filter(col("updated_at") > last_processed)

filtered_df.show()